# Diagnóstico de la capa Gold — validación del JOIN (§2.3.1 paso 2)

**Objetivo:** validar *estáticamente y con datos* el `join` con que se construye la Gold
(`pipeline/02_Medallion_Ecommerce.ipynb`), antes de congelar el esquema (doc 00 §3, §13).

**Regla de cuota (doc 02 §4):** este notebook **NO re-escanea los 14 GB**. Lee la **Silver ya
materializada** y trabaja sobre una **muestra de SESIONES** (no de eventos, para no romper sesiones).
Cachea la muestra y evita full-scans repetidos.

> Es un notebook de **validación**, no de producción: no escribe ninguna tabla.

## 1. Muestra de Silver (por sesión)

Tomamos una fracción de `user_session` distintas y traemos **todos** sus eventos. Así cada sesión
queda íntegra y los conteos de grano/etiqueta son fieles.

In [ ]:
from pyspark.sql import functions as F, Window

silver_table_path = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
df_silver_full = spark.read.format("delta").load(silver_table_path)

FRACCION = 0.02   # ajusta si quieres una muestra mayor/menor
SEED = 42

sesiones = (df_silver_full.select("user_session").distinct()
            .sample(withReplacement=False, fraction=FRACCION, seed=SEED))
df = df_silver_full.join(sesiones, on="user_session", how="inner").cache()

n_sesiones = df.select("user_session").distinct().count()
print(f"Sesiones en la muestra : {n_sesiones:,}")
print(f"Eventos en la muestra  : {df.count():,}")

## 2. Reproducción EXACTA de la lógica actual de la Gold

Idéntica a la celda Gold del Medallion: corte anti-fuga sobre *features*, target sobre la sesión
completa, e `inner join` por `user_session`.

In [ ]:
# --- Corte anti-fuga: solo eventos PREVIOS al primer cart/purchase ---
w = Window.partitionBy("user_session").orderBy("event_time")

df_flags = df.withColumn(
    "is_critical", F.when(F.col("event_type").isin("cart", "purchase"), 1).otherwise(0))

df_pre = (df_flags
          .withColumn("cum_critical", F.sum("is_critical").over(w))
          .filter(F.col("cum_critical") == 0))   # comportamiento previo al corte

# --- Target: ¿la sesión (completa) contiene un purchase? ---
df_target = df.groupBy("user_session").agg(
    F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("target_purchase"))

# --- Features (grano user_session, user_id) sobre los eventos pre-corte ---
df_features = df_pre.groupBy("user_session", "user_id").agg(
    F.count("product_id").alias("total_views"),
    F.countDistinct("product_id").alias("distinct_products_viewed"),
    F.countDistinct("brand").alias("brands_compared"),
    F.countDistinct("macro_category").alias("categories_explored"),
    (F.unix_timestamp(F.max("event_time")) - F.unix_timestamp(F.min("event_time"))).alias("browsing_duration_sec"))

# --- JOIN actual ---
df_gold = df_features.join(df_target, on="user_session", how="inner")

## 3. Diagnósticos

Conteos antes/después del join, tasa de etiqueta, descartes, huérfanas, duplicados y grano.

In [ ]:
n_target    = df_target.count()
tasa_target = df_target.agg(F.avg("target_purchase")).first()[0]

n_feat_rows = df_features.count()
n_feat_sess = df_features.select("user_session").distinct().count()

n_gold_rows = df_gold.count()
n_gold_sess = df_gold.select("user_session").distinct().count()
tasa_gold   = df_gold.agg(F.avg("target_purchase")).first()[0]

# Sesiones DESCARTADAS por el inner (están en target, no en features) y su tasa de compra
descartadas = df_target.join(df_features.select("user_session").distinct(), on="user_session", how="left_anti")
n_desc = descartadas.count()
tasa_desc = descartadas.agg(F.avg("target_purchase")).first()[0] if n_desc else 0.0

# Huérfanas: features sin target (debe ser 0; features ⊆ silver ⊆ target)
huerfanas = df_features.join(df_target, on="user_session", how="left_anti").count()

# Duplicados de sesión en Gold (rompe grano 1 fila = 1 sesión)
n_dup_gold = df_gold.groupBy("user_session").count().filter(F.col("count") > 1).count()

# Grano en origen: user_session con >1 user_id (causa de duplicación en el join)
n_multi_user = (df.groupBy("user_session")
                .agg(F.countDistinct("user_id").alias("nu"))
                .filter(F.col("nu") > 1).count())

print("================ GRANO Y CONTEOS ================")
print(f"Sesiones (muestra)................: {n_sesiones:,}")
print(f"Filas df_target...................: {n_target:,}   | tasa etiqueta (purchase): {tasa_target:.4f}")
print(f"Filas df_features.................: {n_feat_rows:,}   | sesiones distintas: {n_feat_sess:,}")
print(f"Filas df_gold (inner).............: {n_gold_rows:,}   | sesiones distintas: {n_gold_sess:,}")
print()
print("================ SESGO DE SELECCIÓN ================")
print(f"Tasa etiqueta en df_target........: {tasa_target:.4f}")
print(f"Tasa etiqueta en df_gold..........: {tasa_gold:.4f}   <-- si << target, hay fuga de positivos")
print(f"Sesiones DESCARTADAS por el inner.: {n_desc:,} ({(n_desc/n_target*100 if n_target else 0):.1f}% de las sesiones)")
print(f"  -> su tasa de compra............: {tasa_desc:.4f}   <-- si >> target, los descartes son positivos")
print()
print("================ INTEGRIDAD DEL JOIN ================")
print(f"Filas huérfanas (feat sin target).: {huerfanas:,}   (esperado: 0)")
print(f"Sesiones DUPLICADAS en Gold.......: {n_dup_gold:,}   (esperado: 0)")
print(f"Coincide grano (gold_rows==sess)..: {n_gold_rows == n_gold_sess}")
print(f"user_session con >1 user_id.......: {n_multi_user:,}   (>0 rompe el grano vía el join)")

## 4. Mecanismo del sesgo: sesiones que abren con cart/purchase

El corte mantiene `cum_critical == 0` (eventos **previos** al primer crítico). Una sesión cuyo
**primer evento** ya es `cart`/`purchase` no tiene ninguna fila con `cum_critical == 0` → desaparece
de `df_features` → el `inner join` la **elimina**. Esas sesiones son desproporcionadamente
**positivas** (target=1). Aquí lo cuantificamos.

También probamos un *gotcha* de Spark: `sum().over(Window.orderBy(...))` usa por defecto el frame
**RANGE** (no ROWS); con timestamps a segundo, una *vista* en el **mismo segundo** que el primer
evento crítico entra en el mismo grupo RANGE y obtiene `cum_critical = 1` → se excluye aunque sea
comportamiento previo legítimo.

In [ ]:
# Sesiones cuyo PRIMER evento (por tiempo) es crítico -> se pierden en el inner join
primer = df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)
abre_critico = primer.filter(F.col("event_type").isin("cart", "purchase"))
n_abre = abre_critico.count()
tasa_abre = (abre_critico.withColumn("p", F.when(F.col("event_type") == "purchase", 1).otherwise(0))
             .agg(F.avg("p")).first()[0]) if n_abre else 0.0

# Víctimas del frame RANGE: vistas en el mismo segundo que el 1er evento crítico de su sesión
t_crit = (df.filter(F.col("event_type").isin("cart", "purchase"))
          .groupBy("user_session").agg(F.min("event_time").alias("t_crit")))
empate = (df.filter(F.col("event_type") == "view").join(t_crit, "user_session")
          .filter(F.col("event_time") == F.col("t_crit")))
n_empate = empate.select("user_session").distinct().count()

print(f"Sesiones que ABREN con cart/purchase....: {n_abre:,} ({(n_abre/n_sesiones*100 if n_sesiones else 0):.1f}% de las sesiones)")
print(f"  -> su tasa de compra..................: {tasa_abre:.4f}   (vs tasa global {tasa_target:.4f})")
print(f"Sesiones con vista en el MISMO segundo que el 1er crítico (frame RANGE): {n_empate:,}")

## 5. Qué esperar si el join está SANO (criterios de lectura)

| Métrica | Sano | Señal de problema |
|---|---|---|
| **Huérfanas** (feat sin target) | `= 0` siempre | `> 0` ⇒ error de claves |
| **Duplicados en Gold** y `gold_rows == gold_sess` | `0` y `True` | `> 0` / `False` ⇒ grano roto (1 fila ≠ 1 sesión) |
| **user_session con >1 user_id** | `0` | `> 0` ⇒ el `join` por `user_session` duplicará el target |
| **tasa_gold vs tasa_target** | ≈ iguales | `tasa_gold << tasa_target` ⇒ se pierden positivos (sesgo) |
| **Sesiones descartadas** | pocas y con tasa ≈ target (descarte ~aleatorio) | muchas y/o `tasa_desc >> tasa_target` ⇒ descarte enriquecido en compras |
| **Sesiones que abren con cart/purchase** | pocas | muchas y con tasa de compra alta ⇒ confirma el mecanismo del sesgo |

**Conclusión esperada (hipótesis a confirmar):** el `inner join` no es el culpable directo; el problema
es que las *features* se construyen **solo** con eventos previos al primer crítico, así que toda sesión
sin comportamiento previo (típicamente positivos que "abren" con la compra/carrito) **se elimina en
silencio** → la tasa de etiqueta de la Gold queda **sesgada a la baja** y el modelo se entrena sobre una
subpoblación condicionada por "haber navegado antes del carrito". Posible además: duplicación de grano si
`user_session` no es 1:1 con `user_id`, y pérdida de vistas del mismo segundo por el frame RANGE.

## 6. Corrección propuesta (NO aplicar aún — solo propuesta)

> Se deja como propuesta para discutir; **no** se aplica al pipeline hasta validar los diagnósticos.

**(a) No descartar sesiones sin comportamiento previo — preservar el universo de sesiones.**
Partir del universo completo (`df_target`, 1 fila por sesión) y traer las features con `left join`,
rellenando con 0/`False` las sesiones sin eventos pre-corte (y marcándolas con un flag informativo).

```python
# Universo = todas las sesiones (df_target). Features opcionales -> left join + defaults.
df_gold = (df_target
           .join(df_features, on="user_session", how="left")
           .withColumn("sin_navegacion_previa", F.col("total_views").isNull())
           .fillna({"total_views": 0, "distinct_products_viewed": 0,
                    "brands_compared": 0, "categories_explored": 0,
                    "browsing_duration_sec": 0}))
```

**(b) Corte anti-fuga determinista y sin el gotcha de RANGE.**
Calcular el tiempo del primer evento crítico por sesión y quedarse con lo **estrictamente anterior**;
así no depende del frame de la ventana y se decide explícitamente qué hacer con el mismo segundo.

```python
t_crit = (df.filter(F.col("event_type").isin("cart", "purchase"))
          .groupBy("user_session").agg(F.min("event_time").alias("t_crit")))
df_pre = (df.join(t_crit, "user_session", "left")
          .filter(F.col("t_crit").isNull() | (F.col("event_time") < F.col("t_crit"))))
# (< t_crit excluye el mismo segundo; usar <= si se decide incluir la vista del mismo segundo)
```

**(c) Garantizar grano 1 fila = 1 sesión.**
Agrupar las features por `user_session` solamente (llevando `user_id` con `first()`), de modo que el
`join` no pueda multiplicar filas aunque una sesión tenga >1 `user_id`.

```python
df_features = df_pre.groupBy("user_session").agg(
    F.first("user_id", ignorenulls=True).alias("user_id"),
    F.count("product_id").alias("total_views"),
    # ... resto de features ...
)
```

**(d) Verificación tras corregir:** re-correr la sección 3 y comprobar que `tasa_gold ≈ tasa_target`,
`duplicados = 0`, `gold_rows == gold_sess` y `huérfanas = 0`.